# 20. Mô phỏng thời gian thực và đo độ trễ, CPU 1 luồng (GĐ I, task 8.1-8.4)
Phát lại flow của **tập test NF-ToN-IoT-v3** theo thứ tự thời gian, gom cửa sổ 5 s, dựng đồ thị bằng đúng hàm offline (scaler đã fit trên train),
dự đoán bằng mô hình đã đóng băng và ngưỡng của nó. Mô hình chính = mô hình AHP gợi ý (`reports/tables/ahp_result.json`); chạy thêm hybrid nhẹ để so sánh.
Không sửa mô hình, cấu hình, ngưỡng đã đóng băng; không chọn lại mô hình theo kết quả test.

In [ ]:
import glob, json, os, resource, subprocess, sys, time
from contextlib import contextmanager
from pathlib import Path

SMOKE = False                      # True: chỉ 300 cửa sổ đầu để kiểm tra nhanh quy trình
EXTRA_MODELS = ["hybrid_light"]    # chạy thêm để so sánh (hybrid chính đã vượt ngưỡng nhẹ 1 MB / 100.000 tham số)
SEED = 42
WORK = "/kaggle/working"
TMP = "/tmp/det"
REPO = "/tmp/gnn-iot-ids"
URL = "https://github.com/minucedev/gnn-iot-ids.git"
INFO = {"steps_s": {}, "device": "CPU", "gpu": None, "smoke": SMOKE, "seed": SEED}
T_START = time.time()


def peak_ram_gb():
    kb = max(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss, resource.getrusage(resource.RUSAGE_CHILDREN).ru_maxrss)
    return round(kb / 2**20, 2)


def save_info():
    INFO["total_s"] = round(time.time() - T_START, 1)
    INFO["peak_ram_gb"] = peak_ram_gb()
    INFO["gpu_hours"] = 0
    with open(f"{WORK}/run_info.json", "w", encoding="utf-8") as f:
        json.dump(INFO, f, ensure_ascii=False, indent=1)


@contextmanager
def step(name):
    t0 = time.time()
    try:
        yield
    finally:
        INFO["steps_s"][name] = round(time.time() - t0, 1)
        save_info()
        print(f"--- {name}: {INFO['steps_s'][name]} s, RAM đỉnh tới nay {INFO['peak_ram_gb']} GB")


def run(cmd, ok=(0,), log=None, **kw):
    if log:
        with open(log, "w", encoding="utf-8") as f:
            r = subprocess.run(cmd, stdout=f, stderr=subprocess.STDOUT, text=True, **kw)
        print(open(log, encoding="utf-8").read()[-6000:])
    else:
        r = subprocess.run(cmd, capture_output=True, text=True, **kw)
        print(r.stdout[-9000:], end="")
        print(r.stderr[-2500:], end="", file=sys.stderr)
    print(f"[mã thoát {r.returncode}]")
    if r.returncode not in ok:
        raise RuntimeError(f"Lệnh lỗi: {' '.join(cmd)}")
    return r


def find(pattern):
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    assert hits, f"Không thấy {pattern} trong /kaggle/input (đã gắn đủ nguồn dữ liệu chưa?)"
    return hits[0]


with step("clone_va_cai_goi"):
    run(["rm", "-rf", REPO])
    run(["git", "clone", "--depth", "1", URL, REPO])
    INFO["commit"] = subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
    with open(f"{REPO}/requirements.txt") as f:
        reqs = [l.strip() for l in f if l.strip() and not l.startswith("#")]
    keep = [r for r in reqs if r.split("==")[0].lower() not in {"torch", "jupyter", "ipykernel"}]
    run([sys.executable, "-m", "pip", "install", "-q", *keep])
import torch
assert not torch.cuda.is_available() or True
INFO["cpu_count"] = os.cpu_count()
save_info()
print("Commit:", INFO["commit"])

## pytest cho phần mới và phần dựng đồ thị offline (dừng nếu lỗi)

In [ ]:
with step("pytest"):
    r = run([sys.executable, "-m", "pytest", "-q", "tests/test_stream_features.py", "tests/test_realtime_eval.py", "tests/test_ahp.py", "tests/test_build_graph.py", "tests/test_detector_stream.py"], cwd=REPO)
    INFO["pytest"] = r.stdout.strip().splitlines()[-1]
save_info()

## Đường dẫn đầu vào (như notebook 17, chỉ bộ NF-ToN-IoT-v3) và mô hình được AHP gợi ý

In [ ]:
DS = "nf_ton_iot_v3"
OLD = f"{REPO}/reports/kaggle"
SPEC = {DS: dict(data=os.path.dirname(find(f"{DS}/split.json")),
                 ckpt=os.path.dirname(find(f"ckpt/{DS}_gat_s42.pt")),
                 folds=os.path.dirname(os.path.dirname(find(f"models_hybrid/{DS}_s42/lgbm_oofavg_il.pkl"))),
                 light=os.path.dirname(os.path.dirname(find(f"models_light/{DS}_s42/lgbm_n100_l15.pkl"))),
                 baseline_search=f"{OLD}/gnn-iot-ids-05-baselines/baseline_search_{DS}.json",
                 ablation_search=f"{OLD}/gnn-iot-ids-16d-ablation-ton/ablation_search_{DS}.json",
                 gnn_csv=f"{OLD}/gnn-iot-ids-07-train-gnn/results_gnn_{DS}.csv",
                 hybrid_csv=f"{OLD}/gnn-iot-ids-15f-hybrid-il-ton/results_hybrid_oofavg_il_{DS}.csv")}
CFG = dict(dataset=DS, split="test", flows_csv=find("NF-ToN-IoT-v3.csv"), expect_commit=INFO["commit"][:7],
           thresholds=f"{OLD}/gnn-iot-ids-16c-aggregate/thresholds.json",
           thresholds_c={DS: f"{OLD}/gnn-iot-ids-16h-c-threshold/thresholds_c_{DS}.json"}, datasets=SPEC)
for k, v in SPEC[DS].items():
    assert os.path.exists(v), f"{k}: không thấy {v}"
for p in (CFG["thresholds"], *CFG["thresholds_c"].values(), CFG["flows_csv"]):
    assert os.path.exists(p), p
json.dump(CFG, open(f"{WORK}/detector_config.json", "w"), ensure_ascii=False, indent=1)
AHP = json.load(open(f"{REPO}/reports/tables/ahp_result.json", encoding="utf-8"))
MAIN = AHP["recommended"]
MODELS = [MAIN] + [m for m in EXTRA_MODELS if m != MAIN]
INFO["config"], INFO["models"], INFO["ahp_recommended"] = CFG, MODELS, MAIN
save_info()
print("Mô hình AHP gợi ý:", MAIN, "| chạy:", MODELS)

## Chạy bộ phát hiện (một tiến trình mỗi mô hình, CPU 1 luồng) và đánh giá

In [ ]:
Path(TMP).mkdir(parents=True, exist_ok=True)
for i, model in enumerate(MODELS):
    det = f"{TMP}/{model}"
    with step(f"detector_{model}"):
        cmd = [sys.executable, "-I", "scripts/detector.py", "--config", f"{WORK}/detector_config.json", "--model", model, "--seed", str(SEED), "--out", det]
        if SMOKE:
            cmd += ["--max-windows", "300"]
        run(cmd, log=f"{WORK}/detector_{model}_log.txt", cwd=REPO)
    out = WORK if i == 0 else f"{WORK}/extra_{model}"
    with step(f"eval_{model}"):
        run([sys.executable, "-I", "scripts/realtime_eval.py", det, out, out], cwd=REPO)
    for fn in ("windows.csv", "detector_summary.json"):
        os.makedirs(out, exist_ok=True)
        subprocess.run(["cp", f"{det}/{fn}", f"{out}/{fn}"], check=True)
    # realtime_eval ghi hình cùng thư mục với bảng; hình của mô hình chính cũng nằm ở thư mục gốc output
    INFO[f"summary_{model}"] = json.load(open(f"{det}/detector_summary.json", encoding="utf-8"))
    save_info()

## Kết quả

In [ ]:
ev = json.load(open(f"{WORK}/realtime_eval.json", encoding="utf-8"))
lat = json.load(open(f"{WORK}/latency.json", encoding="utf-8"))
INFO["cpu"] = lat["env"]["cpu_model"]
INFO["result"] = {"p95_total_ms": lat["total_ms"]["p95"], "p95_build_ms": lat["build_ms"]["p95"], "p95_infer_ms": lat["infer_ms"]["p95"],
                  "n_windows_measured": lat["n_windows_measured"], "ram": lat["ram"],
                  "online_vs_offline": ev["online_vs_offline"], "false_alarms": ev["false_alarms"],
                  "detection": {k: v for k, v in ev["detection"].items() if k != "episodes"}}
save_info()
print(json.dumps(INFO["result"], indent=1, ensure_ascii=False, default=float))
for r in ev["targets"]:
    print(("ĐẠT" if r["dat"] else "KHÔNG ĐẠT" if r["dat"] is False else "báo cáo"), "|", r["chi_tieu"], "|", r["nguong"], "|", r["gia_tri"])
print(os.listdir(WORK))